# CFTC TFF 2012–2025 年度官方原件采集

冻结 `annual_intake_plan.json` 后仅采集未覆盖的 14 个年度，逐年核验精确代码 209742，原件备份 NAS。不读取市场结果。

In [1]:
import csv,io,json,hashlib,zipfile,subprocess,time,tarfile,tempfile,shlex
from datetime import datetime,timezone
from pathlib import Path
from src.data.nas_market_data import _settings,nas_api_call
root=Path.cwd();card=root/'research/stage1_cftc_nq_tff_source_probe_20261023';raw=root/'data/snapshots/stage1_cftc_nq_tff_source_probe_20261023';raw.mkdir(parents=True,exist_ok=True)
plan=json.loads((card/'annual_intake_plan.json').read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and sha(card/'schema_alias_result.json')==plan['parent_identity_result_sha256']
assert plan['years']==list(range(2012,2026)) and all(not (raw/f'fut_fin_txt_{y}.zip').exists() for y in plan['years'])
state={'status':'started','plan_sha256':sha(card/'annual_intake_plan.json'),'GET_attempts':0,'model_fits':0,'native_accounts':0,'market_outcome_reads':0};(card/'annual_intake_state.json').write_text(json.dumps(state,indent=2)+'\n')
receipts=[];checks=[];failure=None
for y in plan['years']:
 url=plan['exact_url_template'].format(year=y);body=raw/f'fut_fin_txt_{y}.zip';headers=raw/f'fut_fin_txt_{y}.zip.headers';started=datetime.now(timezone.utc).isoformat();state['GET_attempts']+=1;(card/'annual_intake_state.json').write_text(json.dumps(state,indent=2)+'\n')
 cmd=['curl','--http1.1','--noproxy','*','--connect-timeout','10','--max-time','90','--max-filesize','20000000','--fail-with-body','-sS','-L','-D',str(headers),'-o',str(body),'-A','Mozilla/5.0 soxl-qlib-source-research','-w','%{http_code} %{url_effective}',url]
 try:
  run=subprocess.run(cmd,capture_output=True,text=True,timeout=110);parts=run.stdout.rsplit(' ',1);status=int(parts[0]) if len(parts)==2 and parts[0].isdigit() else None
  receipt={'year':y,'url':url,'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'curl_exit_code':run.returncode,'http_status':status,'final_url':parts[-1] if parts else None,'stderr':run.stderr[:300]}
 except Exception as exc:receipt={'year':y,'url':url,'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'transport_exception':str(exc)[:300]}
 receipt.update(raw_sha256=sha(body) if body.exists() else None,raw_bytes=body.stat().st_size if body.exists() else 0,headers_sha256=sha(headers) if headers.exists() else None)
 receipts.append(receipt);(card/'annual_intake_receipts.json').write_text(json.dumps(receipts,indent=2)+'\n')
 if receipt.get('curl_exit_code')!=0 or receipt.get('http_status')!=200 or not body.exists():failure=f'{y}_transport';break
 try:
  with zipfile.ZipFile(body) as z:
   members=[x for x in z.namelist() if x.lower().endswith(('.txt','.csv')) and not x.startswith('__MACOSX/')];assert len(members)==1,(y,members);b=z.read(members[0])
  try:s=b.decode('utf-8-sig')
  except UnicodeDecodeError:s=b.decode('cp1252')
  t=csv.DictReader(io.StringIO(s));cols=t.fieldnames;rows=list(t);assert cols and len(rows)>200
  date_cols=[x for x in plan['date_column_aliases'] if x in cols];assert len(date_cols)==1,(y,date_cols)
  assert all(x in cols for x in plan['required_exact_columns'])
  nq=[x for x in rows if x['CFTC_Contract_Market_Code'].strip()==plan['contract_code'] and all(k in x['Market_and_Exchange_Names'].upper() for k in ('NASDAQ','MINI','CHICAGO MERCANTILE'))];assert len(nq)>=20,(y,len(nq))
  from datetime import date
  dates=[date.fromisoformat(x[date_cols[0]].strip()) for x in nq];assert len(dates)==len(set(dates)) and min(dates).year==max(dates).year==y
  num=lambda x:float(x.replace(',','').strip())
  assert all(num(x['Open_Interest_All'])>0 and num(x['Lev_Money_Positions_Long_All'])>=0 and num(x['Lev_Money_Positions_Short_All'])>=0 for x in nq)
  checks.append({'year':y,'raw_sha256':sha(body),'member':members[0],'date_column':date_cols[0],'NQ_rows':len(nq),'first_report_date':str(min(dates)),'last_report_date':str(max(dates)),'Tuesday_report_share':sum(d.weekday()==1 for d in dates)/len(dates),'market_names':sorted({x['Market_and_Exchange_Names'].strip() for x in nq})})
  (card/'annual_intake_checks.json').write_text(json.dumps(checks,indent=2)+'\n')
 except Exception as exc:failure=f'{y}_schema_{type(exc).__name__}_{str(exc)[:120]}';break
 time.sleep(plan['request']['seconds_between_requests'])
files=[{'name':p.name,'sha256':sha(p),'bytes':p.stat().st_size} for p in sorted(raw.iterdir()) if p.name.startswith('fut_fin_txt_') and 2012<=int(p.name.split('_')[-1].split('.')[0])<=2025]
manifest={'plan_sha256':sha(card/'annual_intake_plan.json'),'files':files,'checks_sha256':sha(card/'annual_intake_checks.json') if (card/'annual_intake_checks.json').exists() else None,'receipts_sha256':sha(card/'annual_intake_receipts.json')};(card/'annual_intake_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
nas=None
if files:
 settings=_settings(root)
 with tempfile.TemporaryDirectory(prefix='cftc-tff-annual-') as td:
  archive=Path(td)/'annual.tar.gz'
  with tarfile.open(archive,'w:gz') as t:
   for x in files:t.add(raw/x['name'],arcname='raw/'+x['name'])
   for name in ('annual_intake_plan.json','annual_intake_manifest.json','annual_intake_receipts.json'):t.add(card/name,arcname=name)
  digest=sha(archive);a=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact'];assert a['sha256']==digest and a['size_bytes']==archive.stat().st_size
  remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
  with tempfile.TemporaryDirectory(prefix='cftc-tff-annual-restore-') as rd:
   restored=Path(rd)/'restore.tar.gz'
   with restored.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,timeout=180)
   assert run.returncode==0 and sha(restored)==digest
   with tarfile.open(restored,'r:gz') as t:
    for x in files:
     member=t.extractfile('raw/'+x['name']);assert member and hashlib.sha256(member.read()).hexdigest()==x['sha256']
  nas={'archive_sha256':digest,'remote_digest_verified':True,'scoped_restore_verified':True,'file_count':len(files)}
result={'status':'all_2012_2025_source_identity_passed_release_and_revision_pending' if not failure and len(checks)==14 and nas else 'bounded_annual_intake_partial_or_failed','failure':failure,'plan_sha256':sha(card/'annual_intake_plan.json'),'GET_attempts':state['GET_attempts'],'checks':checks,'NAS':nas,'historical_release_and_first_public_qualified':False,'model_fits':0,'native_accounts':0,'market_outcome_reads':0};(card/'annual_intake_result.json').write_text(json.dumps(result,indent=2)+'\n');state['status']='completed' if result['status'].startswith('all_') else 'partial_or_failed';(card/'annual_intake_state.json').write_text(json.dumps(state,indent=2)+'\n');print(json.dumps({'status':result['status'],'failure':failure,'GETs':state['GET_attempts'],'years':len(checks),'NAS':nas},indent=2))


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{
  "status": "all_2012_2025_source_identity_passed_release_and_revision_pending",
  "failure": null,
  "GETs": 14,
  "years": 14,
  "NAS": {
    "archive_sha256": "8fb79e0754542869d4edf6be530bfbeab8ce1951e213abc789966fe7c88cd298",
    "remote_digest_verified": true,
    "scoped_restore_verified": true,
    "file_count": 28
  }
}


## 中文结论

2012–2025 年 14 个原件全部通过代码209742身份与字段核验，年度原件和响应头已完成 NAS 远端哈希和指定成员恢复。报告日不是可交易时点；2019与2023与2025补发、2019事后补录仍需首发时钟审查。零收益读取、拟合和账户。完整结论见 `conclusion.md`。